In [1]:
# for package auto reload
%load_ext autoreload
%autoreload 2

# for better rendering of plots in jupyter notebook
%matplotlib inline

In [2]:
import sys
print(sys.executable)

c:\Users\HP\anaconda3\envs\ecl-course-2026\python.exe


In [3]:
# base modules
from pathlib import Path
import polars as pl

# for manipulating data
import numpy as np
import pandas as pd

# for Machine Learning
from sklearn.ensemble import RandomForestRegressor


In [4]:
from eclyon.transforms import (
    add_date_columns,
    change_columns_from_str_to_categorical,
    process_df,
)

In [11]:
path_to_data = Path(r"C:\Users\HP\Desktop\Predicting_Company_Future_ML")

In [12]:
import polars as pl

df_firms = pl.read_parquet(Path(r"C:\Users\HP\Desktop\Predicting_Company_Future_ML\stock-stockunitelegale-parquet.parquet"))
print(df_firms.head(10))

shape: (10, 35)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ siren     ┆ statutDif ┆ unitePurg ┆ dateCreat ┆ … ┆ economieS ┆ societeMi ┆ caractere ┆ activite │
│ ---       ┆ fusionUni ┆ eeUniteLe ┆ ionUniteL ┆   ┆ ocialeSol ┆ ssionUnit ┆ Employeur ┆ Principa │
│ str       ┆ teLegale  ┆ gale      ┆ egale     ┆   ┆ idaireUni ┆ eLegale   ┆ UniteLega ┆ leNAF25U │
│           ┆ ---       ┆ ---       ┆ ---       ┆   ┆ teL…      ┆ ---       ┆ le        ┆ niteLe…  │
│           ┆ str       ┆ bool      ┆ date      ┆   ┆ ---       ┆ str       ┆ ---       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆ str       ┆           ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 000325175 ┆ O         ┆ null      ┆ 2000-09-2 ┆ … ┆ null      ┆ null      ┆ null      ┆ 32.12Y   │
│           ┆           ┆           ┆ 6         ┆   ┆           ┆          

In [11]:
null_map = df_firms.null_count().row(0)
n_rows = df_firms.height

items = list(enumerate(zip(df_firms.columns, df_firms.dtypes, null_map), start=1))

start, end = 0, 20  # promeni npr. na 20, 40 pa 40, 60...
for i, (col, dtype, n_null) in items[start:end]:
    non_null = n_rows - n_null
    pct_null = (n_null / n_rows * 100) if n_rows else 0.0
    print(f"{i:02d}. {col} | type: {dtype} | non-null: {non_null:,} | null: {n_null:,} ({pct_null:.1f}%)")

01. siren | type: String | non-null: 30,280,224 | null: 0 (0.0%)
02. statutDiffusionUniteLegale | type: String | non-null: 30,280,224 | null: 0 (0.0%)
03. unitePurgeeUniteLegale | type: Boolean | non-null: 4,261,004 | null: 26,019,220 (85.9%)
04. dateCreationUniteLegale | type: Date | non-null: 29,108,414 | null: 1,171,810 (3.9%)
05. sigleUniteLegale | type: String | non-null: 3,938,472 | null: 26,341,752 (87.0%)
06. sexeUniteLegale | type: String | non-null: 17,136,113 | null: 13,144,111 (43.4%)
07. prenom1UniteLegale | type: String | non-null: 17,145,519 | null: 13,134,705 (43.4%)
08. prenom2UniteLegale | type: String | non-null: 8,056,917 | null: 22,223,307 (73.4%)
09. prenom3UniteLegale | type: String | non-null: 5,139,508 | null: 25,140,716 (83.0%)
10. prenom4UniteLegale | type: String | non-null: 3,371,228 | null: 26,908,996 (88.9%)
11. prenomUsuelUniteLegale | type: String | non-null: 17,145,519 | null: 13,134,705 (43.4%)
12. pseudonymeUniteLegale | type: String | non-null: 3,30

In [12]:
null_map = df_firms.null_count().row(0)
n_rows = df_firms.height

items = list(enumerate(zip(df_firms.columns, df_firms.dtypes, null_map), start=1))

start, end = 20, 35  # promeni npr. na 20, 40 pa 40, 60...
for i, (col, dtype, n_null) in items[start:end]:
    non_null = n_rows - n_null
    pct_null = (n_null / n_rows * 100) if n_rows else 0.0
    print(f"{i:02d}. {col} | type: {dtype} | non-null: {non_null:,} | null: {n_null:,} ({pct_null:.1f}%)")

21. etatAdministratifUniteLegale | type: String | non-null: 30,280,224 | null: 0 (0.0%)
22. nomUniteLegale | type: String | non-null: 17,150,482 | null: 13,129,742 (43.4%)
23. nomUsageUniteLegale | type: String | non-null: 5,543,176 | null: 24,737,048 (81.7%)
24. denominationUniteLegale | type: String | non-null: 16,334,093 | null: 13,946,131 (46.1%)
25. denominationUsuelle1UniteLegale | type: String | non-null: 4,570,798 | null: 25,709,426 (84.9%)
26. denominationUsuelle2UniteLegale | type: String | non-null: 3,190,888 | null: 27,089,336 (89.5%)
27. denominationUsuelle3UniteLegale | type: String | non-null: 3,187,300 | null: 27,092,924 (89.5%)
28. categorieJuridiqueUniteLegale | type: Int64 | non-null: 30,280,224 | null: 0 (0.0%)
29. activitePrincipaleUniteLegale | type: String | non-null: 30,258,702 | null: 21,522 (0.1%)
30. nomenclatureActivitePrincipaleUniteLegale | type: String | non-null: 30,258,702 | null: 21,522 (0.1%)
31. nicSiegeUniteLegale | type: String | non-null: 30,280,2

In [4]:


lf_fin = pl.scan_parquet(r"C:\Users\HP\Desktop\Predicting_Company_Future_ML\export-detail-bilan.parquet")



In [5]:
schema = lf_fin.collect_schema()
print("Total columns:", len(schema))
for i, (col, dtype) in enumerate(schema.items(), 1):
    print(f"{i}. {col} -> {dtype}")

Total columns: 5
1. siren -> String
2. date_cloture_exercice -> Date
3. type_bilan -> String
4. confidentiality -> String
5. liasse -> Map(String, Int32)


In [7]:
sample = lf_fin.head(3).collect()
val = sample["liasse"][0]
print("Type:", type(val))
print("Content:", val)

Type: <class 'dict'>
Content: {'DJ': 1064, 'DP': 48780, 'FW': 385776, 'LN': 5850813, 'AJ': 540293, 'BI': 0, 'FD': 954, 'LS': 1503473, 'NK': 78273512, 'AP': 3174799, 'DQ': 2839131, 'DX': 148218, 'FE': 0, 'FY': 559989, 'HB': 209176, 'HI': 481014, 'QU': 2368581, 'AU': 196924, 'DB': 7963, 'CI': 0, 'CK': 2005218, 'CG': 0, 'CH': 36923, 'BB-BC': 9347425, 'CD-CE': 12917722, 'DD': 71183, 'EA': 123502, 'GG': -1281259, 'GJ': 39482, 'AN': 2288972, 'AT-AU': 47706, 'UC': 12390, 'GM': 58113, 'ØR': 2563605, 'CJ': 17122824, 'DG': 89549282, 'EC': 4341098, 'FL': 135797, 'AS': 80009, 'CA': 1703757, 'HG': 9499, 'LO': 0, 'BH-BI': 2559, 'CE': 301461, 'DI': -289131, 'FQ': 206166, 'HA': 280517, 'KE': 0, 'QW': 57930, 'AN-AO': 1888651, 'CF': 1404319, 'GK': 28079, '7C': 220031, 'FK': 0, 'HF': 111571, 'ØK': 82226, 'BZ-CA': 529304, 'GP': 745630, 'CO-1A': 97498351, 'CR': 1299000, 'PF': 27044, 'BV': 199270, 'CH-CI': 36923, 'EG': 841098, 'FC': 4623, 'FP': 128933, 'AJ-AK': 513249, 'BJ': 84944350, 'I4': 0, 'LP': 333483,

In [8]:
from collections import Counter
import polars as pl

# 1. Check which balance sheet types exist
print(
    lf_fin.select("type_bilan")
    .drop_nulls()
    .group_by("type_bilan")
    .len()
    .sort("len", descending=True)
    .collect()
)

# 2. Check the most frequent codes inside 'liasse' on a sample of 5,000 rows
sample_codes = lf_fin.select("liasse").head(5000).collect()["liasse"]

freq = Counter()
for row in sample_codes:
  if row is not None:
    if isinstance(row, dict):
      for key in row.keys():
        freq[key] += 1
    else:
      for item in row:
        key = (
            item["key"] if isinstance(item, dict) and "key" in item else item
        )
        freq[key] += 1

print("Top 25 most frequent codes in sample:", freq.most_common(25))

shape: (3, 2)
┌────────────┬─────────┐
│ type_bilan ┆ len     │
│ ---        ┆ ---     │
│ str        ┆ u32     │
╞════════════╪═════════╡
│ C          ┆ 4791633 │
│ S          ┆ 1546390 │
│ K          ┆ 30941   │
└────────────┴─────────┘
Top 25 most frequent codes in sample: [('EE', 4804), ('1A', 4804), ('CO', 4804), ('CK', 4803), ('CJ', 4803), ('DL', 4802), ('DA', 4801), ('EC', 4791), ('BJ', 4757), ('BK', 4757), ('CO-1A', 4734), ('CJ-CK', 4734), ('DI', 4731), ('DX', 4723), ('CA', 4719), ('BZ', 4719), ('BJ-BK', 4688), ('BZ-CA', 4665), ('DD', 4659), ('CG', 4656), ('CF', 4656), ('DY', 4635), ('CF-CG', 4601), ('AU', 4462), ('AT', 4462)]


In [ ]:


codes_to_check = ["DL", "EE", "BJ", "CO", "1A", "DI"]


df_sample = (
    lf_fin.filter(pl.col("type_bilan") == "C")
    .select(["liasse"])
    .head(20000)
    .collect()
)


rows = []
for d in df_sample["liasse"]:
  if isinstance(d, dict):
    rows.append({c: d.get(c) for c in codes_to_check})
  else:
    rows.append({c: None for c in codes_to_check})

df_check = pl.DataFrame(rows)

# 3. Ispis statistike za svaku šifru
for c in codes_to_check:
  s = df_check[c].drop_nulls()
  print(
      f"{c:4s} | non-null: {len(s):5d}/20000 | min: {s.min()} | median:"
      f" {s.median()} | max: {s.max()}"
  )

DL   | non-null: 19994/20000 | min: -1350255415 | median: 1075104.0 | max: 2147483647
EE   | non-null: 19996/20000 | min: -2095 | median: 2565911.0 | max: 2147483647
BJ   | non-null: 19757/20000 | min: -193483648 | median: 1482864.0 | max: 2147483647
CO   | non-null: 19995/20000 | min: -2095 | median: 3718669.0 | max: 2147483647
1A   | non-null: 19995/20000 | min: -212391000 | median: 841772.0 | max: 2147483647
DI   | non-null: 19961/20000 | min: -1357057859 | median: 83495.0 | max: 2147483647


In [15]:


codes_to_check = ["DL", "EE", "BJ", "CO", "1A", "DI"]

# 1. Povuci metapodatke i 'liasse' za prvih 20.000 redova gde je type_bilan == "C"
df_sample = (
    lf_fin.filter(pl.col("type_bilan") == "C")
    .select(["siren", "date_cloture_exercice", "liasse"])
    .head(20000)
    .collect()
)

# 2. Izvuci tražene šifre iz svakog rečnika
rows = []
for d in df_sample["liasse"]:
  if isinstance(d, dict):
    rows.append({c: d.get(c) for c in codes_to_check})
  else:
    rows.append({c: None for c in codes_to_check})

df_check = pl.DataFrame(rows)

# 3. Koliko ima vrednosti == 2147483647 po šifri
for c in codes_to_check:
  cnt_max = (df_check[c] == 2147483647).sum()
  print(f"{c:4s} -> count == 2147483647: {cnt_max}")

# 4. Prvih 5 redova sa siren, datumom i izvučenim šiframa
df_preview = pl.concat(
    [df_sample.select(["siren", "date_cloture_exercice"]), df_check],
    how="horizontal",
)
print(df_preview.head(5))

DL   -> count == 2147483647: 8
EE   -> count == 2147483647: 40
BJ   -> count == 2147483647: 50
CO   -> count == 2147483647: 57
1A   -> count == 2147483647: 19
DI   -> count == 2147483647: 2
shape: (5, 8)
┌───────────┬───────────────────┬──────────┬───────────┬──────────┬───────────┬─────────┬──────────┐
│ siren     ┆ date_cloture_exer ┆ DL       ┆ EE        ┆ BJ       ┆ CO        ┆ 1A      ┆ DI       │
│ ---       ┆ cice              ┆ ---      ┆ ---       ┆ ---      ┆ ---       ┆ ---     ┆ ---      │
│ str       ┆ ---               ┆ i64      ┆ i64       ┆ i64      ┆ i64       ┆ i64     ┆ i64      │
│           ┆ date              ┆          ┆           ┆          ┆           ┆         ┆          │
╞═══════════╪═══════════════════╪══════════╪═══════════╪══════════╪═══════════╪═════════╪══════════╡
│ 005420120 ┆ 2018-12-31        ┆ 90269342 ┆ 97498351  ┆ 84944350 ┆ 102067174 ┆ 4568823 ┆ -289131  │
│ 005420120 ┆ 2021-12-31        ┆ 86469939 ┆ 92582739  ┆ 84956534 ┆ 96128752  ┆ 3546013 ┆